# Fig. 1: Vegetation structure

Panels a,b use shared paired-site contrasts. Panels c,d recalculate the four adjusted contrasts from frozen out-of-fold predictions. Confidence intervals are archived by default; set RUN_BOOTSTRAP=True for the optional historical bootstrap. Every action below is initiated in this notebook; the final cell renders both panel groups.

Default reference mode uses the bundled paired reference inputs. Select raw only after supplying all grid-aligned, dimension-matched 0.01-degree TIFFs. Numerical consistency is checked separately from statistical assumptions.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config/analysis.json").exists())
sys.path.insert(0, str(ROOT / "src"))
from elephant_analysis.runtime import configure_runtime
configure_runtime()
DATA_ROOT = ROOT / "data"
RESULTS_ROOT = ROOT / "results"
FIGURES_ROOT = ROOT / "figures"
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_MODE = "reference"  # "raw" for the harmonized TIFF inputs.
RUN_ANALYSIS = True
RUN_BOOTSTRAP = False  # Optional historical algorithm; see reproducibility status.
N_BOOTSTRAP = 100
PAIRED_PARAMETERS = {"block_size": 25, "minimum_valid_fraction": 0.05, "dem_difference_limit": 200.0, "slope_difference_limit": 10.0}
PAIRED_FILE = RESULTS_ROOT / "paired/paired_results.npz"
SUMMARY_FILE = RESULTS_ROOT / "fig1/vegetation_summary.npz"
ADJUSTED_FILE = RESULTS_ROOT / "covariate/adjusted_contrasts.npz"

In [ ]:
from elephant_analysis.data_inputs import load_inputs
from elephant_analysis.paired_sites import compute_paired
from elephant_analysis.covariate_adjustment import compute_adjusted_contrasts
from elephant_analysis.bootstrap import bootstrap_adjusted_contrasts
from elephant_analysis.figure_statistics import summarize_vegetation_contrasts
from elephant_analysis.artifacts import save_arrays, load_arrays, record_run
if RUN_ANALYSIS:
    paired_inputs = load_inputs("paired", DATA_ROOT, mode=INPUT_MODE)
    prediction_inputs = load_inputs("covariate", DATA_ROOT, mode="precomputed")
    print("Paired input route:", INPUT_MODE)
    print("Adjustment route: frozen observations and out-of-fold predictions")

In [ ]:
if RUN_ANALYSIS:
    paired_result = compute_paired(paired_inputs, PAIRED_PARAMETERS) if INPUT_MODE == "raw" else paired_inputs
    save_arrays(PAIRED_FILE, **paired_result)
    summary_result = summarize_vegetation_contrasts(paired_result)
    save_arrays(SUMMARY_FILE, **summary_result)
    adjustment_result = compute_adjusted_contrasts(prediction_inputs)
    if RUN_BOOTSTRAP:
        bootstrap_result = bootstrap_adjusted_contrasts(prediction_inputs, n_boot=N_BOOTSTRAP, seed=2025)
        save_arrays(RESULTS_ROOT / "covariate/bootstrap_results.npz", **bootstrap_result)
        for key in ("ci_lower", "ci_upper", "ci_status", "bootstrap_recomputed"):
            adjustment_result[key] = bootstrap_result[key]
    save_arrays(ADJUSTED_FILE, **adjustment_result)
    provenance_inputs = [DATA_ROOT / "covariate_adjustment/covariate_inputs.npz", DATA_ROOT / "reference/covariate_reference.npz"]
    if INPUT_MODE == "reference":
        provenance_inputs.append(DATA_ROOT / "reference/paired_reference.npz")
    record_run(RESULTS_ROOT / "fig1/run.json", analysis="Fig1", inputs=provenance_inputs,
               parameters={"input_mode": INPUT_MODE, "paired": PAIRED_PARAMETERS, "run_bootstrap": RUN_BOOTSTRAP, "bootstrap_draws": N_BOOTSTRAP}, mode="points_from_frozen_predictions",
               outputs=[SUMMARY_FILE, ADJUSTED_FILE],
               notes=[("Historical bootstrap rerun; known algorithm caveats retained." if RUN_BOOTSTRAP else "Confidence intervals loaded from archived results; bootstrap not rerun."),
                      "Raw input fingerprints require the separate raw-raster run manifest."])
    print("Results saved; reload them in the next cell.")

In [ ]:
summary = load_arrays(SUMMARY_FILE)
adjusted = load_arrays(ADJUSTED_FILE)
print("Outcome order:", adjusted["outcome_labels"])
print("Adjusted estimates:\n", adjusted["point_estimates"])
print("CI status:", adjusted["ci_status"])
print("The source's KS annotation is a normality diagnostic, not a between-group test.")

In [ ]:
# Original plotting cells copied from Main_figure.ipynb.
# Keep source sizes, font settings, labels, colours, positions and line widths.
EXPORT_PDF = True
EXPORT_DPI = 600
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import kstest, gaussian_kde
from elephant_analysis import array_utils as gc
mpl.rcdefaults()
mpl.rcParams["pdf.fonttype"] = 42

def export_original(figure, stem, dpi=EXPORT_DPI):
    # Match the source savefig contract, including tight bounding boxes.
    FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
    figure.savefig(FIGURES_ROOT / (stem + '.png'), dpi=dpi, bbox_inches='tight')
    if EXPORT_PDF:
        figure.savefig(FIGURES_ROOT / (stem + '.pdf'), bbox_inches='tight')

data = [summary["th_values"], summary["tc_values"]]
# BEGIN ORIGINAL CELL 5

from elephant_analysis import array_utils as gc
from scipy.stats import gaussian_kde
import matplotlib.colors as mcolors
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
fig = plt.figure(figsize=(12,4))
plt.subplots_adjust(wspace=0.25,hspace=0.02)

font = {'family': 'sans-serif',
        'sans-serif': 'Arial',
        'weight': 'normal',
        'size': 18}
plt.rc('font', **font)
cl1 = np.array([(47,85,151)])
cl2 = np.array([(22,142,88)])
cl=[cl1/255,cl2/255]

label1=['TH from Potapov $\mathit{et}$ $\mathit{al.}$','TH from Lang $\mathit{et}$ $\mathit{al.}$']
label2=['TC from Reiner $\mathit{et}$ $\mathit{al.}$','TC from Hansen $\mathit{et}$ $\mathit{al.}$']
label=[label1,label2]
tx=['(m)','(%)']
for i in range(2):
    ax = fig.add_subplot(1,2,i+1)
    x=np.linspace(-15,15,1000)
    mean=np.nanmean(data[i])
    delta_rav=gc.nanravel(data[i])
    kenal=gaussian_kde(delta_rav)
    z=kenal.evaluate(x)
    z_mean=kenal.evaluate(mean)
    ax.plot(x,z,lw=3,color=cl[i], zorder=6)
    ax.fill_between(x,0,z,facecolor=cl[i],alpha=0.2)
    ax.vlines(mean,0,z_mean,lw=2,ls='--',color=cl[i], zorder=7)
    p=float(summary[('th', 'tc')[i] + '_ks_normal_p'])
    mean = np.nanmean(delta_rav)
    se = stats.sem(delta_rav)
    ci_low, ci_high = stats.t.interval(0.95, len(delta_rav)-1, loc=mean, scale=se)
    print(mean, ci_low, ci_high)
    print(mean,p,stats.sem(delta_rav),len(delta_rav))

    if p<0.001:

        ax.text(0.75,0.89, 'p<0.001', fontsize=14,transform = ax.transAxes,color=cl[i])
    elif p<0.01:
        ax.text(0.75,0.89, 'p<0.01', fontsize=14,transform = ax.transAxes,color=cl[i])
    elif p<0.05:
        ax.text(0.75,0.89, 'p<0.05', fontsize=14,transform = ax.transAxes,color=cl[i])
    else:
        ax.text(0.75,0.89, 'p={:.2f}'.format(p), fontsize=14,transform = ax.transAxes,color=cl[i]) 

    if i ==0:
        ax.vlines(0,0,0.3,lw=2,ls='--',color='black')
        ax.set_ylim(0,0.3)
        ax.set_yticks(np.arange(0,0.31,0.1))
        ax.set_xlim(-5,5)
        ax.set_xticks(np.arange(-5,5.1,2.5))
        ax=plt.gca()
        ax.yaxis.set_ticks_position('left')
        ax.spines['left'].set_position(('data',-5.3))
        ax.xaxis.set_ticks_position('bottom')
        ax.spines['bottom'].set_position(('data',-0.04))
        ax.set_xlabel('$\Delta$Tree height (m)',labelpad=10)
    else:
        ax.vlines(0,0,0.08,lw=2,ls='--',color='black')
        ax.set_ylim(0,0.08)
        ax.set_yticks(np.arange(0,0.081,0.02))
        ax.set_xlim(-15,15)
        ax.set_xticks(np.arange(-15,15.1,5))
        ax=plt.gca()
        ax.yaxis.set_ticks_position('left')
        ax.spines['left'].set_position(('data',-15.8))
        ax.xaxis.set_ticks_position('bottom')
        ax.spines['bottom'].set_position(('data',-0.0105))
        ax.set_xlabel('$\Delta$Tree cover (%)',labelpad=10)
    ax.legend(fontsize=14,loc='upper left',frameon=False,handlelength=1)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

    ax.set_ylabel('Probability density',labelpad=10)
    
    ax.xaxis.label.set_size(18)
    ax.xaxis.set_tick_params(width=2)

    ax.tick_params(labelsize=14)
    ax.yaxis.label.set_size(18)
    ax.yaxis.set_tick_params(width=2)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)

    if i ==0:
        ax1=fig.add_axes([0.122,-0,0.35,0.12])
        ax1.set_xlim(-5,5)
        ax1.set_xticks(np.arange(-5,5,2.5))
        ax1.axis('off')
    else:
        ax1=fig.add_axes([0.55,-0,0.35,0.12])
        ax1.set_xlim(-15,15)
        ax1.set_xticks(np.arange(-15,15,5))
        ax1.axis('off')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['left'].set_visible(False)
    ax1.tick_params(bottom=False,top=False, left=False, right=False)

    bplot=ax1.boxplot(data[i],
                      vert=False,
                      whis=(10,90),                
                      widths=0.3,
                      patch_artist=True,
                      showmeans=False,
                      meanprops = {'marker':'o','markerfacecolor':cl[i],"markeredgecolor":cl[i],"markersize":15,"alpha":0},
                      medianprops={'linewidth':'0.5',"color":cl[i],"alpha":0},
                      boxprops={"facecolor": 'none', "edgecolor": cl[i],"linewidth":2,"alpha":1},
                      capprops=None,
                      whiskerprops={'linewidth':'2','color':cl[i]},
                      showcaps=False,
                      showfliers = False,
                      flierprops = {'marker':'+','markersize':'3','markeredgecolor':cl[i],'color':cl[i]})
# END ORIGINAL CELL 5
export_original(fig, 'Fig1ab')
plt.show()

# Convert safe arrays to the original plotting cell's dictionary interface.
keys = ['ATT-Match', 'ATE-IPW', 'ATE-Gcomp', 'ATE-DR']
estimates_th = dict(zip(keys, adjusted['point_estimates'][0]))
estimates_tc = dict(zip(keys, adjusted['point_estimates'][1]))
cis_th = dict(zip(keys, zip(adjusted['ci_lower'][0], adjusted['ci_upper'][0])))
cis_tc = dict(zip(keys, zip(adjusted['ci_lower'][1], adjusted['ci_upper'][1])))
# BEGIN ORIGINAL CELL 7
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

label_to_key = {
    "Matching": "ATT-Match",
    "IPW": "ATE-IPW",
    "Standardization": "ATE-Gcomp",
    "DR": "ATE-DR",}
labels = list(label_to_key.keys())
y = np.arange(len(labels))

def prep_data(estimates, cis):
    means, xerr_lo, xerr_hi = [], [], []
    for lab in labels:
        key = label_to_key[lab]
        est = estimates.get(key, np.nan)
        lo, hi = cis.get(key, (np.nan, np.nan))
        left  = max(est - lo, 0.0) if np.isfinite(est) and np.isfinite(lo) else 0.0
        right = max(hi - est, 0.0) if np.isfinite(est) and np.isfinite(hi) else 0.0
        means.append(est)
        xerr_lo.append(left)
        xerr_hi.append(right)
    return np.array(means), [xerr_lo, xerr_hi]

means_tc, xerr_tc = prep_data(estimates_tc, cis_tc)
means_th, xerr_th = prep_data(estimates_th, cis_th)

fig, axes = plt.subplots(1, 2, figsize=(12,4), sharey=True)
plt.subplots_adjust(wspace=0.2,hspace=0.02)
font = {'family': 'sans-serif', 'sans-serif': 'Arial', 'weight': 'normal', 'size': 16}
plt.rc('font', **font)

c_th_marker = (47/255, 85/255, 151/255)  
c_th_error  = (176/255, 191/255, 216/255) 
c_tc_marker = (22/255, 142/255, 88/255) 
c_tc_error  = (177/255, 217/255, 199/255)

ax1 = axes[0]

ax1.errorbar(means_th, y, xerr=xerr_th, fmt="o", mfc=c_th_marker, mec=c_th_marker, ms=10, ecolor=c_th_error, lw=5, capsize=0, barsabove=True)
ax1.axvline(0, linestyle="--", color="black", linewidth=2)
ax1.set_xlim(-2,4)
ax1.set_xticks(np.arange(-2,4.01,2))
ax1.set_xlabel("Effects of elephants on tree height (m)", labelpad=10)
ax1.tick_params(labelsize=18)
ax1.xaxis.label.set_size(18)
ax1.xaxis.set_tick_params(width=1.5)

ax1.spines['top'].set_linewidth(1.5)
ax1.spines['right'].set_linewidth(1.5)
ax1.spines['left'].set_linewidth(1.5)
ax1.spines['bottom'].set_linewidth(1.5)
ax1.set_ylim(min(y)-0.5, max(y)+0.5)

ax2 = axes[1]
ax2.errorbar(means_tc, y, xerr=xerr_tc, fmt="o",mfc=c_tc_marker, mec=c_tc_marker, ms=10, ecolor=c_tc_error, lw=5, capsize=0, barsabove=True)
ax2.axvline(0, linestyle="--", color="black", linewidth=2)
ax2.set_xlim(-5,10)
ax2.set_xticks(np.arange(-5,10.01,5))
ax2.set_yticks(y)
ax2.set_yticklabels(labels)
ax2.set_xlabel("Effects of elephants on tree cover (%)", labelpad=10)
ax2.tick_params(labelsize=18)
ax2.xaxis.label.set_size(18)
ax2.xaxis.set_tick_params(width=1.5)

ax2.spines['top'].set_linewidth(1.5)
ax2.spines['right'].set_linewidth(1.5)
ax2.spines['left'].set_linewidth(1.5)
ax2.spines['bottom'].set_linewidth(1.5)
ax2.set_ylim(min(y)-0.5, max(y)+0.5)
# END ORIGINAL CELL 7
export_original(fig, 'Fig1cd')
plt.show()
